# 日前共同失误与物理信息：冻结样本复核
离线研究，不写生产；下面单元在交付前已执行。完整实验协议、排除原因与研究限制见 REPORT.md。


In [1]:
import json, sys
from pathlib import Path
base = Path.cwd()
if not (base / 'samples.json').exists():
    base = base / 'docs/research/p0_physical_20260920'
sys.path.insert(0, str(base.resolve()))
from study import Guard
guard = Guard()
from evaluate import partition, validate_evidence
from model import scored_rows, summarize
samples = json.loads((base / 'samples.json').read_text())
validate_evidence(samples)
print('配对站日', len(samples), '主样本', sum(s['primary_eligible'] for s in samples))


配对站日 1414 主样本 1037


In [2]:
train, validation, refit, outer, split = partition(samples)
print(json.dumps(split, ensure_ascii=False, indent=2))


{
  "validation_fit_cutoff": "2026-09-04T11:01:13.475000+00:00",
  "outer_fit_cutoff": "2026-09-11T11:05:08.361000+00:00",
  "train_n": 600,
  "validation_n": 130,
  "refit_n": 756,
  "outer_n": 257,
  "validation_unavailable_labels": 24,
  "outer_days": [
    "2026-09-12",
    "2026-09-13",
    "2026-09-14",
    "2026-09-15",
    "2026-09-16",
    "2026-09-17",
    "2026-09-18",
    "2026-09-19"
  ],
  "outer_stations": [
    "CYYZ",
    "EDDM",
    "EFHK",
    "EHAM",
    "EPWA",
    "FACT",
    "LEMD",
    "LFPB",
    "LIMC",
    "LTAC",
    "MMMX",
    "MPMG",
    "NZWN",
    "OEJN",
    "OPKC",
    "RCSS",
    "RJTT",
    "RKPK",
    "RKSI",
    "RPLL",
    "SAEZ",
    "SBGR",
    "VILK",
    "WMKK",
    "WSSS",
    "ZBAA",
    "ZGGG",
    "ZGSZ",
    "ZHHH",
    "ZSPD",
    "ZSQD",
    "ZUCK",
    "ZUUU"
  ]
}


In [3]:
artifacts = json.loads((base / 'candidate_artifacts.json').read_text())
for arm, artifact in artifacts['artifacts'].items():
    scored = scored_rows(outer, artifact)
    print(arm, summarize(scored))


identity {'n': 257, 'days': 8, 'stations': 33, 'crps_c': 0.555333961740499, 'zero_hits': 0, 'rmse_c': 1.090720607867892, 'mae_c': 0.8218226670778533, 'bias_c': 0.04060265472131903, 'nll': 1.4075993973204741, 'nll_status': 'FINITE', 'coverage80': 0.9377431906614786, 'finite_width80_c': 3.7448559670781894, 'open_interval80_share': 0.054474708171206226, 'coverage90': 0.9649805447470817, 'finite_width90_c': 4.556521739130435, 'open_interval90_share': 0.10505836575875487}
joint {'n': 257, 'days': 8, 'stations': 33, 'crps_c': 0.5511177127052467, 'zero_hits': 0, 'rmse_c': 1.0851181655157607, 'mae_c': 0.8133536760400781, 'bias_c': 0.08698449135428847, 'nll': 1.3932149588728282, 'nll_status': 'FINITE', 'coverage80': 0.914396887159533, 'finite_width80_c': 3.411522633744856, 'open_interval80_share': 0.054474708171206226, 'coverage90': 0.9610894941634242, 'finite_width90_c': 4.252136752136752, 'open_interval90_share': 0.08949416342412451}
tx {'n': 257, 'days': 8, 'stations': 33, 'crps_c': 0.551550

In [4]:
result = json.loads((base / 'results.json').read_text())
print('联合对 P0', result['outer']['joint']['paired_crps'])
print('联合对 TX', result['joint_vs_tx'])
print('中心误差分解', result['attribution_outer'])


联合对 P0 {'crps_change_pct': -0.7592276586214863, 'date_block_1_ci95_pct': [-2.3363513102143574, 0.7535366297484195], 'date_block_2_ci95_pct': [-2.2841039365461224, 0.6457385788339431]}
联合对 TX {'crps_change_pct': -0.07842153564406873, 'date_block_1_ci95_pct': [-0.6027060657642438, 0.4181440966249557], 'date_block_2_ci95_pct': [-0.5716973333695258, 0.36946481160053946]}
中心误差分解 {'groups': {'common_cold': {'cloud_mean': 0.38125375264355527, 'mean_native_error_c': -1.0573730813333333, 'n': 70, 'precip_mean': 0.04012098958122986, 'wind_mean_mps': 3.691773065246614}, 'common_warm': {'cloud_mean': 0.3319543299166587, 'mean_native_error_c': 1.3221506141316746, 'n': 73, 'precip_mean': 0.09845750006104559, 'wind_mean_mps': 3.8546230149053167}, 'inside_center_hull': {'cloud_mean': 0.35400277854435996, 'mean_native_error_c': 0.008887845464859967, 'n': 114, 'precip_mean': 0.06642736930647572, 'wind_mean_mps': 3.2982020107071954}}, 'interpretation': 'Descriptive, not causal; per-model centers, not ful

In [5]:
audit = json.loads((base / 'source_audit.json').read_text())
reconcile = json.loads((base / 'source_reconciliation.json').read_text())
print('参考物理字段', audit['reference_physics'])
print('真值映射', reconcile['counts'])
assert not reconcile['clock_failures']
print('未来回执失败', len(reconcile['clock_failures']))


参考物理字段 {'captured_at_or_after_day': 1677, 'captured_before_day': 18, 'forward': 1695, 'rows': 1695}
真值映射 {'asos_awc': 304, 'asos_bucket_mismatch': 91, 'asos_madis_hfmetar': 73, 'asos_value_mismatch': 337, 'primary_awc': 1035, 'primary_missing': 2}
未来回执失败 0


In [6]:
native = json.loads((base / 'native_physics_inventory.json').read_text())
print('原生历史包盘点', native['counts'])
print('证据边界', native['qualification'])
assert native['production_writes'] == 0
assert native['network_requests'] == 0


原生历史包盘点 {'candidate_rejection:curve_missing': 238, 'candidate_rejection:receipt_mismatch': 75, 'candidate_rejection:run_mismatch': 5999, 'matched_unique_system_scopes': 2370, 'packet_captures': 16570, 'packet_curves': 15607, 'packets': 437, 'root_curve_missing': 91, 'root_curve_present': 1323, 'samples': 1414, 'samples_all_systems_matched': 739, 'samples_any_system_matched': 1220, 'system_scopes': 3482}
证据边界 Scope/run/receipt/hash audit only. Does not recertify training artifacts, raw decoding, publication time, member paths, or formal OOS.
